# HPO smoke test — Colab and CPU

This lightweight notebook validates the additions without duplicating framework code. It uses tiny synthetic 32×32 data, the repository model builders and training loop, and bounded studies. GPU is used when available; CPU is a supported fallback.

Coverage keywords: Python dictionary, CSV, Proxy, Successive halving, Full, Continuous, Resume, Pareto.

## 1. Setup and dependency checks

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil

REPO_URL = "https://github.com/TrueRottweiler/WashingtonCsed504.git"
BRANCH = "feature/hpo-framework"  # change to the branch containing these additions
REPO_ROOT = Path("/content/WashingtonCsed504")
USE_EXISTING_CHECKOUT = (REPO_ROOT / "src/a1-cv/hpo").exists()

if not USE_EXISTING_CHECKOUT:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)], check=True)

CV_DIR = REPO_ROOT / "src/a1-cv"
os.chdir(CV_DIR)
if str(CV_DIR) not in sys.path:
    sys.path.insert(0, str(CV_DIR))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "hpo_requirements.txt"], check=True)
import torch, optuna, hpo
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Optuna:", optuna.__version__)
print("HPO package:", hpo.__file__)

## 2. Hardware detection and resource plan

In [ ]:
from hpo.hardware import detect_hardware
from hpo.scheduler import plan_resources
hardware = detect_hardware()
plan = plan_resources(hardware, device="auto", requested_concurrency=1)
print(json.dumps(hardware.to_dict(), indent=2, default=str))
print(json.dumps(plan.to_dict(), indent=2, default=str))

## 3. Search-space inputs: Python dictionary, list, CSV, and manual input

In [ ]:
from hpo.search_space import normalize_space, load_csv, combination_count, preview_rows
from hpo.notebook_api import preview_dataframe, optional_widgets

DICT_SPACE = {
    "optimizer": {"type": "categorical", "choices": ["sgd", "adamw"], "default": "sgd"},
    "learning_rate": {"type": "float", "low": 1e-4, "high": 0.2, "log": True, "default": 0.01},
    "batch_size": {"type": "categorical", "choices": [16, 32], "default": 16},
    "momentum": {"type": "float", "low": 0.8, "high": 0.95, "step": 0.05, "default": 0.9, "condition": 'optimizer == "sgd"'},
    "beta1": {"type": "float", "low": 0.8, "high": 0.95, "step": 0.05, "default": 0.9, "condition": 'optimizer == "adamw"'},
}
LIST_SPACE = [{"name": name, **spec} for name, spec in DICT_SPACE.items()]
MANUAL_SPACE = DICT_SPACE.copy()  # edit this normal Python cell
CSV_PATH = CV_DIR / "hpo_configs/search_spaces/resnet18_cifar.csv"

spaces = {
    "dictionary": normalize_space(DICT_SPACE, source_name="notebook dictionary"),
    "list": normalize_space(LIST_SPACE, source_name="notebook list"),
    "manual": normalize_space(MANUAL_SPACE, source_name="manual notebook input"),
    "csv": load_csv(CSV_PATH),
}
for name, specs in spaces.items():
    print(name, "finite combinations:", combination_count(specs))
    display(preview_dataframe(specs))

widgets = optional_widgets(DICT_SPACE)
if widgets is not None:
    display(widgets)

## 4. Repository model and dataset adapters

In [ ]:
import torch
from hpo.adapters import RepoModules, build_trial_model, build_trial_dataset
modules = RepoModules(REPO_ROOT)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bundle = build_trial_dataset(modules, {"name":"synthetic","train_examples":64,"validation_examples":32,"test_examples":32,"num_classes":10}, {"seed":42}, device)
resnet = build_trial_model(modules, {"name":"resnet18"}, bundle.num_classes, device)
vit = build_trial_model(modules, {"name":"vit","hidden_dim":48,"layers":1,"heads":3,"mlp_dim":96,"patch_size":8}, bundle.num_classes, device)
print("Dataset:", bundle.name, bundle.train_examples, bundle.validation_examples, bundle.test_examples)
print("ResNet output:", tuple(resnet(next(bundle.train.epoch(2, True))[0]).shape))
print("ViT output:", tuple(vit(next(bundle.train.epoch(2, True))[0]).shape))

## 5. Batch and runtime calibration

In [ ]:
from hpo.calibration import calibrate_batch_sizes
calibration = calibrate_batch_sizes(
    lambda: build_trial_model(modules, {"name":"vit","hidden_dim":48,"layers":1,"heads":3,"mlp_dim":96,"patch_size":8}, 10, device),
    lambda batch: (torch.randn(batch,3,32,32,device=device), torch.randint(0,10,(batch,),device=device)),
    device=device,
    candidates=[2,4,8],
    precision="fp16" if device.type == "cuda" else "fp32",
    warmup_steps=1,
    measure_steps=1,
)
print(json.dumps(calibration.to_dict(), indent=2))

## 6. Tiny proxy, successive-halving, full, and continuous studies

In [ ]:
required_names = ["REPO_ROOT", "CV_DIR", "device"]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Run Sections 1 through 5 first. "
        f"Missing variables: {missing_names}"
    )

print("Sections 1–5 are ready.")
print("Repository:", REPO_ROOT)
print("CV directory:", CV_DIR)
print("Device:", device)

In [ ]:
from pathlib import Path
import copy
import json
import shutil
import yaml

from hpo.config import load_study_config
from hpo.study import HpoStudy


# Fail with an actionable message instead of a NameError.
required_names = ["REPO_ROOT", "CV_DIR", "device"]
missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Run Sections 1 through 5 before Section 6. "
        f"Missing variables: {missing_names}"
    )


# Remove only earlier smoke-test outputs.
OUTPUT_ROOT = Path("/content/hpo_smoke_outputs")
shutil.rmtree(OUTPUT_ROOT, ignore_errors=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


# Load the repository smoke configuration.
BASE = yaml.safe_load(
    (
        CV_DIR
        / "hpo_configs/smoke/resnet18_cifar10.yaml"
    ).read_text()
)


# Use a tiny synthetic dataset and tiny ViT.
BASE["dataset"] = {
    "name": "synthetic",
    "train_examples": 64,
    "validation_examples": 32,
    "test_examples": 32,
    "num_classes": 10,
}

BASE["model"] = {
    "name": "vit",
    "hidden_dim": 48,
    "layers": 1,
    "heads": 3,
    "mlp_dim": 96,
    "patch_size": 8,
}

BASE["runtime"].update(
    {
        "device": device.type,
        "precision": (
            "fp16"
            if device.type == "cuda"
            else "fp32"
        ),
        "concurrent_trials": 1,
        "intraop_threads": 1,
        "interop_threads": 1,
    }
)


# IMPORTANT:
# Parameters belong directly under search_space.
# There must not be an extra "inline" wrapper.
BASE["search_space"] = {
    "optimizer": {
        "type": "fixed",
        "default": "adamw",
    },
    "learning_rate": {
        "type": "categorical",
        "choices": [0.001, 0.003],
        "default": 0.001,
    },
    "batch_size": {
        "type": "fixed",
        "default": 16,
    },
    "weight_decay": {
        "type": "fixed",
        "default": 0.01,
    },
}


BASE["proxy"].update(
    {
        "trials": 2,
        "promote_top_k": 1,
        "budget": {
            "epochs": 1,
            "max_steps": 1,
            "data_fraction": 1.0,
            "validation_fraction": 1.0,
        },
    }
)

BASE["successive_halving"].update(
    {
        "rung_budgets": [2],
        "minimum_trials_per_rung": 1,
    }
)

BASE["full"].update(
    {
        "trials": 1,
        "budget": {
            "epochs": 1,
            "max_steps": 1,
            "data_fraction": 1.0,
            "validation_fraction": 1.0,
            "seeds": [42],
        },
    }
)

BASE["continuous"].update(
    {
        "enabled": False,
        "maximum_trials": 2,
        "maximum_wall_time_hours": 0.05,
        "stop_after_no_improvement_trials": 2,
    }
)


def make_smoke_config(
    study_name: str,
    mode: str,
) -> dict:
    """Build one isolated smoke-study configuration."""

    raw = copy.deepcopy(BASE)

    raw["study"].update(
        {
            "name": study_name,
            "output_dir": str(OUTPUT_ROOT),
            "storage_path": str(
                OUTPUT_ROOT / f"{study_name}.db"
            ),
        }
    )

    # The parser gives search.mode priority over a top-level mode.
    raw.setdefault("search", {})["mode"] = mode

    return raw


# Validate the search-space structure before training.
preflight_raw = make_smoke_config(
    "smoke-preflight",
    "proxy",
)

preflight_config = load_study_config(
    preflight_raw
)

parameter_names = [
    spec.name
    for spec in preflight_config.search_space
]

print("Normalized parameters:", parameter_names)

assert parameter_names == [
    "optimizer",
    "learning_rate",
    "batch_size",
    "weight_decay",
]

assert preflight_config.mode == "proxy"

print("Search-space preflight: PASSED")


# Run the three explicit modes.
SUMMARIES = {}

for mode in [
    "proxy",
    "successive_halving",
    "full",
]:
    raw = make_smoke_config(
        f"smoke-{mode}",
        mode,
    )

    validated_config = load_study_config(raw)

    print(
        f"Starting {mode}: "
        f"validated mode={validated_config.mode}"
    )

    assert validated_config.mode == mode

    study = HpoStudy(
        validated_config,
        repo_root=REPO_ROOT,
    )

    SUMMARIES[mode] = study.run()


# Run bounded continuous proxy mode.
continuous_raw = make_smoke_config(
    "smoke-continuous",
    "proxy",
)

continuous_raw["continuous"]["enabled"] = True
continuous_raw["continuous"]["strategy"] = "proxy"

continuous_config = load_study_config(
    continuous_raw
)

assert continuous_config.mode == "proxy"
assert continuous_config.continuous.enabled

print(
    "Starting continuous: "
    f"base mode={continuous_config.mode}, "
    f"strategy={continuous_config.continuous.strategy}"
)

SUMMARIES["continuous"] = HpoStudy(
    continuous_config,
    repo_root=REPO_ROOT,
).run()


print(
    json.dumps(
        SUMMARIES,
        indent=2,
        default=str,
    )
)

## 7. Persistence, resumption, pruning, Pareto, and export

In [ ]:
from hpo.persistence import read_jsonl
from hpo.reporting import export_reports
from hpo.config import load_study_config
from hpo.schemas import ObjectiveSpec

sh_dir=OUTPUT_ROOT/"smoke-successive_halving"
events=read_jsonl(sh_dir/"events.jsonl")
assert any(e.get("event")=="candidate_pruned" for e in events), "halving did not record pruning"
# Reopen the completed study; completed work must not be duplicated.
raw = copy.deepcopy(BASE)

raw["study"].update(
    {
        "name": "smoke-successive_halving",
        "output_dir": str(OUTPUT_ROOT),
        "storage_path": str(
            OUTPUT_ROOT
            / "smoke-successive_halving.db"
        ),
    }
)

raw.setdefault(
    "search",
    {},
)["mode"] = "successive_halving"
before=len(read_jsonl(sh_dir/"trials.jsonl")); resumed=HpoStudy(raw, repo_root=REPO_ROOT).run(); after=len(read_jsonl(sh_dir/"trials.jsonl"))
assert before==after
report=export_reports(sh_dir,[ObjectiveSpec("validation_top1","maximize",True),ObjectiveSpec("wall_seconds","minimize",False)])
print("Resume summary:", resumed)
print("Report:", report)
print("Exports:", sorted(p.name for p in sh_dir.iterdir()))

## 8. Final pass/fail summary

In [ ]:
checks={
"imports": True,
"hardware": hardware is not None,
"list_input": bool(spaces["list"]),
"dictionary_input": bool(spaces["dictionary"]),
"csv_input": bool(spaces["csv"]),
"manual_input": bool(spaces["manual"]),
"model_adapter": resnet is not None and vit is not None,
"dataset_adapter": bundle.train_examples==64,
"calibration": calibration.largest_fitting_batch is not None,
"proxy": "proxy" in SUMMARIES,
"successive_halving": "successive_halving" in SUMMARIES,
"full": "full" in SUMMARIES,
"continuous": "continuous" in SUMMARIES,
"pruning": any(e.get("event")=="candidate_pruned" for e in events),
"resume_no_duplicates": before==after,
"pareto_export": (sh_dir/"pareto_trials.csv").exists(),
}
print(json.dumps(checks, indent=2))
assert all(checks.values())
print("FINAL RESULT: PASS")